# Isolated V2 Mem Adapter: F0 (No-Memory) & F1 (Memory) Training on Kaggle

Notebook này được cấu hình tối ưu cho **Kaggle 2x T4 GPUs** nhằm huấn luyện cặp adapter đối chứng:
- **F0 (`no_memory`)**: Base model + LoRA trained on demonstration tokens under identity memory prompt (baseline ablation).
- **F1 (`memory`)**: Base model + LoRA trained on memory-conditioned demonstration tokens under full perception events.

### Tính năng chính:
1. **Prebuilt Shards Support (Fast-Path)**: Tự động nhận diện dataset shards `libero_mem_t1_t3_shards_v2` trong `/kaggle/input` (`/kaggle/input/libero-mem-t1-t3-shards-v2/dataset`), bỏ qua 100% phase chuẩn bị dữ liệu (tiết kiệm ~5 giờ).
2. **Modular Training Controls**: Tùy biến chạy độc lập hoặc tuần tự cả hai mô hình qua `RUN_TRAIN_F0` và `RUN_TRAIN_F1`.
3. **Auto-Resume Riêng Biệt**: Cơ chế kiểm tra `checkpoint-*` độc lập cho từng output (`output_f0`, `output_f1`), tự động resume nếu phiên bị timeout/ngắt.
4. **Strict Interface Validation**: Tự động chạy `validate_memory_sft_export.py` sau khi hoàn thành mỗi adapter để kiểm tra tính toàn vẹn inference/weights.

### Hướng dẫn thiết lập trên Kaggle:
1. **GPU**: Chọn **2x T4 GPUs** (Accelerator: GPU T4 x2).
2. **Internet**: Bật **Internet On** (để pip install lerobot/peft/accelerate và tải backbone assets).
3. **Inputs cần gắn vào Notebook**:
   - **Dataset Shards (Khuyên dùng)**: `luquanglinh/libero-mem-t1-t3-shards-v2` (chứa `dataset/shards/` và `dataset/manifest.json`).
   - **Mã nguồn**: `mem_adapter_v1` (chứa thư mục `mem_adapter_v1_code` hoặc file zip).


In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, zipfile, tempfile

# ==============================================================================
# ⚙️ CẤU HÌNH THI CÔNG & QUẢN LÝ HUẤN LUYỆN F0 / F1 (KAGGLE 12-HOUR SESSION)
# ==============================================================================
# - Chế độ Fast-Path (Khuyên dùng): RUN_DATA_PREP = False (dùng shards có sẵn, bỏ qua 5h prep)
# - Chế độ Chuẩn bị: RUN_DATA_PREP = True (tạo perception memory từ raw data ~4.5h)
RUN_DATA_PREP = False         # False khi dùng dataset shards đã chuẩn bị sẵn (tiết kiệm ~5h)
RUN_TRAIN_F0 = True           # Huấn luyện F0: No-Memory Baseline (--condition no_memory)
RUN_TRAIN_F1 = True           # Huấn luyện F1: Full-Memory Policy (--condition memory)
TRAIN_STEPS = 5000            # Số optimizer steps cho mỗi condition (khoảng 3.5h/condition trên 2x T4)
SAVE_EVERY = 500              # Tần suất lưu checkpoint (mỗi 500 steps)
AUTO_RESUME_TRAINING = True   # Tự động tìm checkpoint cao nhất của từng condition để resume
GPUS = 2                      # Accelerate 2x T4 GPUs

INPUT = Path('/kaggle/input')
SCRATCH = Path('/kaggle/temp/mem-adapter-v1')
WORK = Path('/kaggle/working/mem-adapter-v1')
PREP = WORK / 'preparation'
DATA = WORK / 'dataset'
BASE_ROOT = SCRATCH / 'base_demonstrations'
ASSETS = SCRATCH / 'assets'
REPO = SCRATCH / 'code'

EXPECTED_LOCK = '8275e468382835c06f9cc738361df26b7452965850878e7ca882f392c9ca4165'
PROCESSING_RESERVE_GIB = 50

for d in (SCRATCH, WORK, PREP, BASE_ROOT, ASSETS, REPO):
    d.mkdir(parents=True, exist_ok=True)

def run(python, *args):
    subprocess.run([str(python), *map(str, args)], cwd=REPO, check=True)

def digest(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def extract_checked(archive, destination):
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as z:
        for member in z.infolist():
            target_path = (destination / member.filename).resolve()
            if not target_path.is_relative_to(destination.resolve()):
                raise ValueError(f'Archive path escapes destination: {member.filename}')
        z.extractall(destination)

# Khởi tạo mã nguồn vào REPO
# Ưu tiên 1: Tải mã nguồn mới nhất từ GitHub nếu có kết nối Internet
if not (REPO / 'scripts/train_mem_adapter.py').is_file() or not (REPO / '.git').is_dir():
    try:
        print('-> Đang đồng bộ mã nguồn mới nhất từ GitHub...')
        shutil.rmtree(REPO, ignore_errors=True)
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Linhcodelinhtinh/Thesis.git', str(REPO)], check=True)
        print('✓ Đã đồng bộ repo từ GitHub thành công!')
    except Exception as e:
        print(f'Không thể clone từ GitHub ({e}), chuyển sang đọc từ /kaggle/input...')

if not (REPO / 'scripts/train_mem_adapter.py').is_file():
    code_zips = sorted(INPUT.rglob('*code*.zip')) + sorted(INPUT.rglob('mem_adapter*.zip'))
    found = False
    for cz in code_zips:
        try:
            with zipfile.ZipFile(cz) as z:
                if any('train_mem_adapter.py' in n for n in z.namelist()):
                    print(f'Extracting code bundle: {cz} -> {REPO}...')
                    extract_checked(cz, REPO)
                    found = True
                    break
        except Exception:
            continue
    if not found:
        candidates = [p.parents[1] for p in INPUT.rglob('scripts/train_mem_adapter.py')]
        for c in candidates:
            if (c / 'scripts/train_mem_adapter.py').is_file():
                print(f'Copying code from {c} -> {REPO}...')
                shutil.copytree(c, REPO, dirs_exist_ok=True)
                found = True
                break
    assert (REPO / 'scripts/train_mem_adapter.py').is_file(), 'Không tìm thấy source code trong /kaggle/input!'

# Fallback Hotfix: Đảm bảo src/training/mixed_sft.py hỗ trợ predicted_rgb_relations_v1 nếu dùng bundle cũ
mixed_sft_file = REPO / 'src/training/mixed_sft.py'
if mixed_sft_file.is_file():
    code_text = mixed_sft_file.read_text(encoding='utf-8')
    if 'predicted_rgb_relations_v1' not in code_text:
        print('-> Phát hiện code bundle cũ (chưa có predicted_rgb_relations_v1). Đang cập nhật hotfix...')
        code_text = code_text.replace(
            'PREDICTED_MEMORY_SOURCES = {"rgb_cycle_v3": "predicted_rgb_v3"}',
            'PREDICTED_MEMORY_SOURCES = {"rgb_cycle_v3": "predicted_rgb_v3", "rgb_relations_v1": "predicted_rgb_relations_v1"}'
        )
        code_text = code_text.replace(
            '("none", "predicted_rgb_v3")',
            '{"none", *PREDICTED_MEMORY_SOURCES.values()}'
        )
        code_text = code_text.replace(
            'manifest["memory_source"] == "predicted_rgb_v3"',
            'manifest["memory_source"] in PREDICTED_MEMORY_SOURCES.values()'
        )
        mixed_sft_file.write_text(code_text, encoding='utf-8')
        print('✓ Đã cập nhật hotfix cho mixed_sft.py!')

print('✓ Source code đã sẵn sàng tại:', REPO)


In [ ]:
import subprocess, sys, shutil, os
from pathlib import Path

ENV_DIR = SCRATCH / 'perception-env'
PERCEPTION_PY = ENV_DIR / 'bin/python'
TRAIN_PY = sys.executable
PIP_BIN = ENV_DIR / 'bin/pip'

if not RUN_DATA_PREP:
    print('RUN_DATA_PREP = False. Bỏ qua khởi tạo perception-env (không cần thiết khi dùng shards có sẵn).')
else:
    # Khởi tạo perception-env chuẩn bằng virtualenv (độc lập, chứa Transformers 4.51.3)
    if not PERCEPTION_PY.is_file():
        print('1. Đang tạo perception-env bằng virtualenv...')
        shutil.rmtree(ENV_DIR, ignore_errors=True)
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'virtualenv'], check=True)
        subprocess.run([sys.executable, '-m', 'virtualenv', str(ENV_DIR)], check=True)
        subprocess.run([str(PIP_BIN), 'install', '-q', 'hf_xet'], check=True)

        print('2. Đang cài đặt thư viện vào perception-env...')
        clean_env = dict(os.environ, PYTHONNOUSERSITE='1')
        req_path = REPO / 'envs/memory_perception.txt'
        subprocess.run([
            str(PIP_BIN), 'install', '--no-cache-dir', '-r', str(req_path)
        ], check=True, env=clean_env)

    print('3. Kiểm tra xác nhận perception-env...')
    clean_env = dict(os.environ, PYTHONNOUSERSITE='1')
    subprocess.run([
        str(PERCEPTION_PY), '-c',
        'import transformers, torch; assert transformers.__version__ == "4.51.3"; print("✓ Perception-env sẵn sàng! Transformers:", transformers.__version__, "| CUDA:", torch.cuda.is_available())'
    ], check=True, env=clean_env)


## 1. Xác thực nguồn dữ liệu từ 2 Datasets & Quản lý dung lượng
Khi `RUN_DATA_PREP = True`, notebook liên kết 10 demonstrations base từ Dataset 1 và trích xuất derived mem demonstrations từ Dataset 2. Khi dùng shards có sẵn (`RUN_DATA_PREP = False`), bước tải raw demonstrations được tự động bỏ qua.


In [ ]:
import json

if RUN_DATA_PREP:
    # 1. Liên kết 10 demonstrations base từ Dataset 1 trong /kaggle/input
    print('1. Đang kiểm tra và liên kết Base LIBERO-10 demonstrations từ Dataset 1...')
    run(sys.executable, REPO / 'scripts/download_memory_training_sources.py',
        '--base-only', '--base-root', BASE_ROOT, '--input-root', INPUT,
        '--report', WORK / 'base-sources.json')
    base_report = json.loads((WORK / 'base-sources.json').read_text())
    attached_base = sum(1 for s in base_report['sources'] if s['origin'] in ('attached', 'existing'))
    downloaded_base = sum(1 for s in base_report['sources'] if s['origin'] == 'download')
    print(f'✓ Đã chuẩn bị {len(base_report["sources"])}/10 task base demonstrations (từ input: {attached_base}, tải từ HF: {downloaded_base})!')

    # 2. Định vị và giải nén Derived Mem demonstrations từ Dataset 2
    print('2. Đang kiểm tra Derived Mem demonstrations từ Dataset 2...')
    derived_zips = sorted(INPUT.rglob('*derived*.zip'))
    if derived_zips:
        print(f'-> Tìm thấy derived archive: {derived_zips[0]}')
        MEM_BUNDLE = SCRATCH / 'derived_sources'
        if not (MEM_BUNDLE / 'derived_raw').is_dir():
            print(f'-> Đang giải nén derived sources sang {MEM_BUNDLE}...')
            extract_checked(derived_zips[0], MEM_BUNDLE)
        MEM_ROOT = MEM_BUNDLE / 'derived_raw'
        SPLIT_LOCK = MEM_BUNDLE / 'split.lock.json'
    else:
        candidates = [p.parent for p in INPUT.rglob('KITCHEN_SCENE1_1_pick_up_the_bowl_and_place_it_back_on_the_plate_demo.hdf5')]
        assert candidates, 'Không tìm thấy derived sources (zip hoặc folder) trong /kaggle/input!'
        MEM_ROOT = candidates[0]
        lock_cands = list(MEM_ROOT.glob('split.lock.json')) + list(MEM_ROOT.parent.glob('split.lock.json'))
        SPLIT_LOCK = lock_cands[0] if lock_cands else (REPO / 'resources/manifests/libero_mem_t1_t3_base10_split_v2.json')

    lock_content = json.loads(SPLIT_LOCK.read_text(encoding='utf-8'))
    actual_lock = lock_content.get('lock_sha256') or digest(SPLIT_LOCK)
    assert actual_lock == EXPECTED_LOCK, f'Split lock mismatch! Nhận: {actual_lock}, Mong đợi: {EXPECTED_LOCK}'
    print(f'✓ Đã xác thực Derived Mem split lock ({EXPECTED_LOCK[:12]}...)!')
else:
    print('RUN_DATA_PREP = False. Bỏ qua bước nạp raw base và derived demonstrations.')

# 3. Kiểm tra dung lượng đĩa
free_scratch = shutil.disk_usage(SCRATCH).free / 2**30
free_work = shutil.disk_usage(WORK).free / 2**30
print(f'Dung lượng trống: Scratch = {free_scratch:.1f} GiB | Working = {free_work:.1f} GiB')


## 2. Cài đặt thư viện huấn luyện và tải assets cần thiết


In [ ]:
# Cài đặt các thư viện phục vụ cho phase huấn luyện trên Kaggle kernel
!pip uninstall -y torchao
!pip install -q "lerobot[smolvla,peft,dataset]==0.6.1" "peft>=0.12.0" "accelerate>=0.34.0" "h5py>=3.10.0" "huggingface-hub>=0.25.0"


In [ ]:
# Tải checkpoint gốc và tokenizer đã pin revision
run(TRAIN_PY, REPO / 'scripts/memory_sft_assets.py', 'fetch', ASSETS)
if RUN_DATA_PREP:
    run(PERCEPTION_PY, REPO / 'scripts/download_memory_detector.py',
        '--output', REPO / 'resources/models/grounding-dino-tiny')


## 3. Khôi phục dữ liệu đã tạo & Hoàn thành phase chuẩn bị dữ liệu (Auto-Resume & Parallel)
- **Song song hóa**: Chạy đồng thời T1 trên GPU 0 và T3 trên GPU 1 (giảm thời gian xuống ~4.5h).
- **Auto-Resume**: Nếu có tiến trình dở dang, tự động phục hồi từng episode đã hoàn tất mà không phải chạy lại từ đầu.
- **Bỏ qua khi đã có Shards**: Nếu dataset shards hoàn chỉnh đã tồn tại (ở session này hoặc được gắn từ session trước / dataset đính kèm), tự động bỏ qua toàn bộ bước này trong 0 giây.


In [ ]:
from concurrent.futures import ThreadPoolExecutor

if not RUN_DATA_PREP:
    print('RUN_DATA_PREP = False. Đang định vị dataset shards trong /kaggle/input...')

    # 1. Kiểm tra các đường dẫn chuẩn theo cấu trúc Kaggle Dataset đính kèm:
    # Cấu trúc: /kaggle/input/libero-mem-t1-t3-shards-v2/dataset (chứa shards/ và manifest.json)
    direct_candidates = [
        INPUT / 'libero-mem-t1-t3-shards-v2/dataset',
        INPUT / 'libero-mem-t1-t3-shards-v2',
        DATA,
    ]
    for dc in direct_candidates:
        if (dc / 'manifest.json').is_file():
            try:
                m = json.loads((dc / 'manifest.json').read_text(encoding='utf-8'))
                if m.get('artifact') == 'mixed_base_mem_sft_shards' and m.get('split_lock_sha256') == EXPECTED_LOCK:
                    DATA = dc
                    print(f'-> Gắn kết trực tiếp dataset shards từ: {DATA}')
                    break
            except Exception:
                continue

    # 2. Quét rglob('manifest.json') nếu tên dataset trên Kaggle bị mount với tên khác
    if not (DATA / 'manifest.json').is_file():
        for cand in INPUT.rglob('manifest.json'):
            try:
                m = json.loads(cand.read_text(encoding='utf-8'))
                if m.get('artifact') == 'mixed_base_mem_sft_shards' and m.get('split_lock_sha256') == EXPECTED_LOCK:
                    DATA = cand.parent
                    print(f'-> Phát hiện dataset shards qua rglob tại: {DATA}')
                    break
            except Exception:
                continue

    # 3. Fallback nếu dataset ở dạng file zip chưa giải nén:
    if not (DATA / 'manifest.json').is_file():
        shard_zips = sorted(INPUT.rglob('*shards*.zip')) + sorted(INPUT.rglob('*mem*.zip'))
        for sz in shard_zips:
            try:
                with zipfile.ZipFile(sz) as z:
                    if any('manifest.json' in n for n in z.namelist()):
                        extract_target = SCRATCH / 'dataset'
                        print(f'-> Đang giải nén {sz.name} sang {extract_target}...')
                        extract_checked(sz, extract_target)
                        for cand in extract_target.rglob('manifest.json'):
                            m = json.loads(cand.read_text(encoding='utf-8'))
                            if m.get('artifact') == 'mixed_base_mem_sft_shards':
                                DATA = cand.parent
                                break
                        if (DATA / 'manifest.json').is_file():
                            break
            except Exception:
                continue
else:
    PREP.mkdir(parents=True, exist_ok=True)

    # 1. Tự động khôi phục dữ liệu preparation nếu có từ session trước trong /kaggle/input
    # Cấu trúc: preparation/preparation/memory_T1 hoặc rglob('memory_T1')
    if not (PREP / 'memory_T1/manifest.json').is_file():
        for cand in INPUT.rglob('memory_T1'):
            if cand.is_dir() and (cand / 'manifest.json').is_file():
                parent_prep = cand.parent
                print(f'-> Khôi phục preparation artifacts từ: {parent_prep} -> {PREP}')
                for item in parent_prep.iterdir():
                    dest = PREP / item.name
                    if not dest.exists():
                        if item.is_dir():
                            shutil.copytree(item, dest)
                        else:
                            shutil.copy2(item, dest)
                break

    # 2. Chạy Perception song song trên 2x GPU T4 (T1 trên GPU 0, T3 trên GPU 1)
    protocol = json.loads((REPO / 'configs/benchmarks/libero_mem_t1_t3_protocol.json').read_text())
    tasks = [(task['alias'], task['name']) for task in protocol['tasks']]
    DETECTOR = REPO / 'resources/models/grounding-dino-tiny'

    def prepare_task(item):
        idx, (alias, task_name) = item
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(idx % GPUS), PYTHONNOUSERSITE='1')
        out_dir = PREP / ('memory_' + alias)
        print(f'-> [GPU {idx % GPUS}] Đang xử lý task {alias} ({task_name})...')
        subprocess.run([
            str(PERCEPTION_PY), '-u', str(REPO / 'scripts/prepare_libero_mem_sft.py'),
            '--source-kind', 'derived_raw', '--data-dir', str(MEM_ROOT),
            '--split-lock', str(SPLIT_LOCK), '--task-name', task_name,
            '--tokenizer-path', str(ASSETS / 'backbone'), '--model-dir', str(DETECTOR),
            '--device', 'cuda', '--output-dir', str(out_dir), '--resume',
        ], cwd=REPO, env=env, check=True)
        print(f'✓ [GPU {idx % GPUS}] Hoàn thành perception task {alias}!')

    print('Bắt đầu xử lý perception song song 2 GPU (giảm thời gian xuống ~4-5h)...')
    with ThreadPoolExecutor(max_workers=GPUS) as pool:
        list(pool.map(prepare_task, enumerate(tasks)))

    # 3. Đóng gói mixed dataset shards (10 base + T1/T3 mem)
    if not (DATA / 'manifest.json').is_file():
        print('Đang đóng gói mixed base + mem shards vào working directory...')
        with tempfile.TemporaryDirectory(dir=SCRATCH) as temp_d:
            stage = Path(temp_d) / 'dataset'
            run(PERCEPTION_PY, REPO / 'scripts/prepare_mixed_memory_sft.py',
                '--split-lock', str(SPLIT_LOCK), '--base-root', str(BASE_ROOT), '--mem-root', str(MEM_ROOT),
                '--tokenizer-path', str(ASSETS / 'backbone'), '--output-dir', str(stage),
                '--memory-index', *(PREP / ('memory_' + alias) for alias, _ in tasks))
            shutil.copytree(stage, DATA, dirs_exist_ok=True)

assert (DATA / 'manifest.json').is_file(), 'Chưa tìm thấy dataset shards! Hãy gắn dataset shards vào /kaggle/input hoặc đặt RUN_DATA_PREP = True.'
# Xác thực toàn vẹn dataset
manifest = json.loads((DATA / 'manifest.json').read_text(encoding='utf-8'))
lock = json.loads((DATA / 'split.lock.json').read_text(encoding='utf-8'))
assert lock.get('lock_sha256') == EXPECTED_LOCK, f'Split lock trong dataset không khớp EXPECTED_LOCK ({EXPECTED_LOCK[:12]}... vs {lock.get("lock_sha256")[:12]}...)!'
run(TRAIN_PY, REPO / 'scripts/validate_mixed_memory_sft.py', '--data-dir', DATA)
shutil.copy2(DATA / 'manifest.json', WORK / 'dataset-manifest.json')
shutil.copy2(DATA / 'split.lock.json', WORK / 'dataset-split.lock.json')
print('✓ Dataset đã sẵn sàng cho training tại:', DATA)
print('Counts:', manifest['counts'])


## 4. Huấn luyện Adapter (Fine-tuning phase với Auto-Resume & 2 Điều kiện F0/F1)
Khởi chạy huấn luyện phân tán 2x T4 qua Accelerate với batch size hiệu dụng 32 (batch 2 x accumulation 8 x 2 GPUs):
1. **F0 (`no_memory`)**: Điều kiện đối chứng không có memory prompt (giữ nguyên demo tokens, xóa perception context).
2. **F1 (`memory`)**: Điều kiện memory đầy đủ (hướng dẫn + temporal memory context).

Cả hai điều kiện đều được chạy với seed 42, 5,000 steps, checkpoint lưu mỗi 500 steps và hỗ trợ tự động resume độc lập nếu phiên bị ngắt.


In [ ]:
def find_latest_checkpoint(output_dir, expected_steps):
    if not AUTO_RESUME_TRAINING:
        return None
    ckpt_candidates = sorted(output_dir.glob('checkpoint-*')) + sorted(INPUT.rglob(f'{output_dir.name}/checkpoint-*'))
    valid_ckpts = []
    for c in ckpt_candidates:
        if c.is_dir() and (c / 'progress.json').is_file() and (c / 'model.safetensors').is_file():
            try:
                step_num = int(c.name.split('-')[-1])
                valid_ckpts.append((step_num, c))
            except ValueError:
                continue
    if valid_ckpts:
        valid_ckpts.sort(key=lambda x: x[0])
        latest_step, latest_dir = valid_ckpts[-1]
        if latest_step < expected_steps:
            print(f'-> [AUTO-RESUME] Phát hiện checkpoint cho {output_dir.name}: {latest_dir.name} (step {latest_step}/{expected_steps}). Tiếp tục!')
            return latest_dir
        else:
            print(f'-> Checkpoint {latest_dir.name} cho {output_dir.name} đã hoàn thành toàn bộ {expected_steps} steps!')
    return None

# Cập nhật recipe base cho số steps và tần suất lưu checkpoint
recipe_path = REPO / 'configs/training/mem_adapter_v1.json'
recipe = json.loads(recipe_path.read_text(encoding='utf-8'))
recipe['steps'] = TRAIN_STEPS
recipe['save_every'] = SAVE_EVERY
recipe_path.write_text(json.dumps(recipe, indent=2) + '\n', encoding='utf-8')
print(f'-> Cấu hình training: {TRAIN_STEPS} steps, lưu checkpoint mỗi {SAVE_EVERY} steps.\n')

training_runs = []
if RUN_TRAIN_F0:
    training_runs.append(('no_memory', WORK / 'output_f0', 'F0 (No-Memory Baseline)'))
if RUN_TRAIN_F1:
    training_runs.append(('memory', WORK / 'output_f1', 'F1 (Full-Memory Policy)'))

if not training_runs:
    print('Cả RUN_TRAIN_F0 và RUN_TRAIN_F1 đều là False. Bỏ qua phase huấn luyện.')
else:
    for condition, output_dir, label in training_runs:
        print('=' * 75)
        print(f'🚀 BẮT ĐẦU HUẤN LUYỆN: {label} [--condition {condition}]')
        print('=' * 75)
        output_dir.mkdir(parents=True, exist_ok=True)
        final_export = output_dir / 'pretrained_model'

        if final_export.is_dir() and (final_export / 'adapter_model.safetensors').is_file():
            print(f'✓ {label} đã hoàn tất và xuất khẩu tại {final_export}. Bỏ qua training.\n')
        else:
            resume_ckpt = find_latest_checkpoint(output_dir, TRAIN_STEPS)
            train_args = [
                str(TRAIN_PY), str(REPO / 'scripts/train_mem_adapter.py'),
                '--data-dir', str(DATA),
                '--assets-dir', str(ASSETS),
                '--output-dir', str(output_dir),
                '--condition', condition,
                '--gpus', str(GPUS),
                '--steps', str(TRAIN_STEPS),
                '--save-every', str(SAVE_EVERY),
            ]
            if resume_ckpt:
                train_args += ['--resume', str(resume_ckpt)]

            print(f'Khởi chạy: {" ".join(train_args)}')
            subprocess.run(train_args, cwd=REPO, check=True)
            print(f'✓ Huấn luyện hoàn tất: {label}!\n')

        # Xác thực tính toàn vẹn adapter export
        print(f'Đang kiểm tra tính toàn vẹn export adapter của {label}...')
        run(TRAIN_PY, REPO / 'scripts/validate_memory_sft_export.py',
            '--checkpoint', output_dir / 'pretrained_model',
            '--base-checkpoint', ASSETS / 'policy',
            '--data-dir', DATA,
            '--device', 'cuda')
        print(f'✓ Kiểm tra export {label} thành công!\n')

    saved_bytes = sum(p.stat().st_size for p in WORK.rglob('*') if p.is_file())
    print('=' * 75)
    print(f'🎉 TẤT CẢ CÁC ĐIỀU KIỆN HUẤN LUYỆN ĐÃ HOÀN TẤT! Dung lượng working: {saved_bytes / 1e9:.2f} GB')
    print('=' * 75)


## 5. Lưu trữ Artifacts và Hướng dẫn Quản lý Phiên (12h Kaggle Session)

### Khuyên dùng: Quy trình chạy an toàn:
1. **Gắn Dataset Shards**: Gắn `luquanglinh/libero-mem-t1-t3-shards-v2` và `mem_adapter_v1` làm Input.
2. **Cấu hình Fast-Path**: Giữ `RUN_DATA_PREP = False` để bỏ qua 100% thời gian chuẩn bị dữ liệu (tiết kiệm ~5 giờ).
3. **Huấn luyện cả 2 model**: Đặt `RUN_TRAIN_F0 = True` và `RUN_TRAIN_F1 = True`. Notebook sẽ huấn luyện tuần tự F0 rồi F1 (~3.5h mỗi model trên 2x T4 GPU, tổng ~7h, hoàn toàn trọn vẹn trong 12h của Kaggle).
4. **Cơ chế Auto-Resume**: Nếu phiên bị ngắt, lần chạy tiếp theo sẽ tự động nhận diện `checkpoint-*` cao nhất của từng model trong working directory để tiếp tục mà không mất công train lại từ đầu.

### Xuất và tải Adapter về máy Local:
- **F0 Adapter**: Nằm tại `/kaggle/working/mem-adapter-v1/output_f0/pretrained_model`
- **F1 Adapter**: Nằm tại `/kaggle/working/mem-adapter-v1/output_f1/pretrained_model`
Tải hai thư mục này về máy local để đặt vào `experiments/training/` và tiến hành rollout benchmarking trên mô phỏng LIBERO!
